<a href="https://colab.research.google.com/github/beingAnujChaudhary/DSFS-Joel-Grus/blob/main/notebooks/chapter_01_introduction/01_introduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/DSFS-Joel-Grus.git

# Move into repository
%cd /content/DSFS-Joel-Grus

# Move into Chapter 1 folder
%cd notebooks/chapter_01_introduction

Mounted at /content/drive
Cloning into 'DSFS-Joel-Grus'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 64 (delta 35), reused 47 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 820.35 KiB | 3.42 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/DSFS-Joel-Grus
/content/DSFS-Joel-Grus/notebooks/chapter_01_introduction


In [1]:
# Chapter 1: Introduction
# Data Science from Scratch - Joel Grus

# Environment check
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("✅ Environment ready!")
print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")

✅ Environment ready!
NumPy version: 2.0.2
pandas version: 2.2.2


# Chapter 1: Introduction — DataSciencester Network
**Book**: *Data Science from Scratch* by Joel Grus  
**Focus**: First-principles data analysis using basic Python tools

This notebook implements all examples from Chapter 1. We'll explore a hypothetical social network called "DataSciencester" to learn foundational data science techniques without relying on heavy libraries.

In [4]:
# Environment & Setup
from collections import Counter, defaultdict

print("✅ Chapter 1 environment ready!")
print("Python division is already float (Python 3+). No need for __future__ import.")

✅ Chapter 1 environment ready!
Python division is already float (Python 3+). No need for __future__ import.


## 1. The DataSciencester Network
We start with a list of users and their friendship connections.

In [8]:
# Network Data Setup
users = [
    {"id": 0, "name": "Hero"}, {"id": 1, "name": "Dunn"},
    {"id": 2, "name": "Sue"},  {"id": 3, "name": "Chi"},
    {"id": 4, "name": "Thor"}, {"id": 5, "name": "Clive"},
    {"id": 6, "name": "Hicks"}, {"id": 7, "name": "Devin"},
    {"id": 8, "name": "Kate"}, {"id": 9, "name": "Klein"}
]


friendships = [
    (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
    (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)
]

# Augment each user dict with a list of friends
for user in users:
    user["friends"] = []

for i, j in friendships:
    users[i]["friends"].append(users[j])
    users[j]["friends"].append(users[i])

# Verify structure
print(f"User 0 (Hero) friends: {[f['name'] for f in users[0]['friends']]}")

User 0 (Hero) friends: ['Dunn', 'Sue']


## 2. Finding Key Connectors (Degree Centrality)
We calculate how many friends each user has to identify central nodes in the network.

In [9]:
# Degree Centrality
def number_of_friends(user):
    """Returns the number of friends a user has."""
    return len(user["friends"])

total_connections = sum(number_of_friends(user) for user in users)
num_users = len(users)
avg_connections = total_connections / num_users

print(f"Total connections: {total_connections}")
print(f"Number of users: {num_users}")
print(f"Average connections per user: {avg_connections:.2f}")

# Sort users by number of friends (most to least)
num_friends_by_id = [(user["id"], number_of_friends(user)) for user in users]
sorted_by_friends = sorted(num_friends_by_id, key=lambda id_friends: id_friends[1], reverse=True)

print("\nUsers ranked by number of friends:")
for user_id, count in sorted_by_friends:
    print(f"User {user_id} ({users[user_id]['name']}): {count} friends")

Total connections: 24
Number of users: 10
Average connections per user: 2.40

Users ranked by number of friends:
User 1 (Dunn): 3 friends
User 2 (Sue): 3 friends
User 3 (Chi): 3 friends
User 5 (Clive): 3 friends
User 8 (Kate): 3 friends
User 0 (Hero): 2 friends
User 4 (Thor): 2 friends
User 6 (Hicks): 2 friends
User 7 (Devin): 2 friends
User 9 (Klein): 1 friends


## 3. Data Scientists You May Know
We find mutual friends using `Counter` and filter out existing friends/self.

In [10]:
# Friends of Friends Logic
def not_the_same(user, other_user):
    """Two users are not the same if they have different IDs."""
    return user["id"] != other_user["id"]

def not_friends(user, other_user):
    """other_user is not a friend if not in user's friends list."""
    return all(not_the_same(friend, other_user) for friend in user["friends"])

def friends_of_friend_ids(user):
    """Counts mutual friends, excluding self and existing friends."""
    return Counter(
        foaf["id"]
        for friend in user["friends"]
        for foaf in friend["friends"]
        if not_the_same(user, foaf) and not_friends(user, foaf)
    )

# Test on Chi (id 3)
chi = users[3]
print(f"Friends of friend counts for {chi['name']} (id {chi['id']}):")
print(friends_of_friend_ids(chi))

Friends of friend counts for Chi (id 3):
Counter({0: 2, 5: 1})


## 4. Interest-Based Recommendations
We build indexes to efficiently find users with shared interests.

In [13]:
# Interests Indexing
interests = [
    (0, "Hadoop"), (0, "Big Data"), (0, "HBase"), (0, "Java"),
    (0, "Spark"), (0, "Storm"), (0, "Cassandra"),
    (1, "NoSQL"), (1, "MongoDB"), (1, "Cassandra"), (1, "HBase"),
    (1, "Postgres"), (2, "Python"), (2, "scikit-learn"), (2, "scipy"),
    (2, "numpy"), (2, "statsmodels"), (2, "pandas"), (3, "R"), (3, "Python"),
    (3, "statistics"), (3, "regression"), (3, "probability"),
    (4, "machine learning"), (4, "regression"), (4, "decision trees"),
    (4, "libsvm"), (5, "Python"), (5, "R"), (5, "Java"), (5, "C++"),
    (5, "Haskell"), (5, "programming languages"), (6, "statistics"),
    (6, "probability"), (6, "mathematics"), (6, "theory"),
    (7, "machine learning"), (7, "scikit-learn"), (7, "Mahout"),
    (7, "neural networks"), (8, "neural networks"), (8, "deep learning"),
    (8, "Big Data"), (8, "artificial intelligence"), (9, "Hadoop"),
    (9, "Java"), (9, "MapReduce"), (9, "Big Data")
]

# Build fast lookup indexes
user_ids_by_interest = defaultdict(list)
interests_by_user_id = defaultdict(list)

for user_id, interest in interests:
    user_ids_by_interest[interest].append(user_id)
    interests_by_user_id[user_id].append(interest)

def most_common_interests_with(user):
    """Finds users with the most shared interests."""
    return Counter(
        interested_user_id
        for interest in interests_by_user_id[user["id"]]
        for interested_user_id in user_ids_by_interest[interest]
        if interested_user_id != user["id"]
    )

hero = users[0]
print(f"Users with most shared interests with {hero['name']} (id {hero['id']}):")
print(most_common_interests_with(hero))

Users with most shared interests with Hero (id 0):
Counter({9: 3, 1: 2, 8: 1, 5: 1})


## 5. Salaries and Experience (Bucketing & Aggregation)
Raw averages can be misleading. We group data into buckets to reveal patterns.

In [14]:
# Salary vs Tenure Analysis
salaries_and_tenures = [
    (83000, 8.7), (88000, 8.1), (48000, 0.7), (76000, 6),
    (69000, 6.5), (76000, 7.5), (60000, 2.5), (83000, 10),
    (48000, 1.9), (63000, 4.2)
]

def tenure_bucket(tenure):
    if tenure < 2:
        return "less than two"
    elif tenure < 5:
        return "between two and five"
    else:
        return "more than five"

# Group salaries by tenure bucket
salary_by_tenure_bucket = defaultdict(list)
for salary, tenure in salaries_and_tenures:
    bucket = tenure_bucket(tenure)
    salary_by_tenure_bucket[bucket].append(salary)

# Compute average salary per bucket
average_salary_by_bucket = {
    bucket: sum(salaries) / len(salaries)
    for bucket, salaries in salary_by_tenure_bucket.items()
}

print("Average salary by tenure bucket:")
for bucket, avg_sal in average_salary_by_bucket.items():
    print(f"  {bucket}: ${avg_sal:,.2f}")

Average salary by tenure bucket:
  more than five: $79,166.67
  less than two: $48,000.00
  between two and five: $61,500.00


## 6. Paid Accounts (Rule-Based Prediction)
A simple heuristic model based on experience thresholds.

In [15]:
# Simple Prediction Rule
def predict_paid_or_unpaid(years_experience):
    if years_experience < 3.0:
        return "paid"
    elif years_experience < 8.5:
        return "unpaid"
    else:
        return "paid"

# Test the rule against actual data
print("Predictions vs Actual (Experience, Predicted, Actual):")
experience_paid_actual = [
    (0.7, "paid"), (1.9, "unpaid"), (2.5, "paid"), (4.2, "unpaid"),
    (6.0, "unpaid"), (6.5, "unpaid"), (7.5, "unpaid"), (8.1, "unpaid"),
    (8.7, "paid"), (10.0, "paid")
]

for exp, actual in experience_paid_actual:
    predicted = predict_paid_or_unpaid(exp)
    match = "✅" if predicted == actual else "❌"
    print(f"  {exp:4.1f} yrs | Predicted: {predicted:6} | Actual: {actual:6} {match}")

Predictions vs Actual (Experience, Predicted, Actual):
   0.7 yrs | Predicted: paid   | Actual: paid   ✅
   1.9 yrs | Predicted: paid   | Actual: unpaid ❌
   2.5 yrs | Predicted: paid   | Actual: paid   ✅
   4.2 yrs | Predicted: unpaid | Actual: unpaid ✅
   6.0 yrs | Predicted: unpaid | Actual: unpaid ✅
   6.5 yrs | Predicted: unpaid | Actual: unpaid ✅
   7.5 yrs | Predicted: unpaid | Actual: unpaid ✅
   8.1 yrs | Predicted: unpaid | Actual: unpaid ✅
   8.7 yrs | Predicted: paid   | Actual: paid   ✅
  10.0 yrs | Predicted: paid   | Actual: paid   ✅


## 7. Topics of Interest (Word Frequency)
Extracting popular topics by tokenizing and counting interest strings.

In [16]:
# Word Frequency Analysis
words_and_counts = Counter(
    word
    for user, interest in interests
    for word in interest.lower().split()
)

print("Words appearing more than once in interests:")
for word, count in words_and_counts.most_common():
    if count > 1:
        print(f"  {word}: {count}")

Words appearing more than once in interests:
  big: 3
  data: 3
  java: 3
  python: 3
  learning: 3
  hadoop: 2
  hbase: 2
  cassandra: 2
  scikit-learn: 2
  r: 2
  statistics: 2
  regression: 2
  probability: 2
  machine: 2
  neural: 2
  networks: 2


## 📝 Chapter 1 Summary
- **Degree Centrality**: Simple count of connections, but can miss intuitive "key connectors".
- **Mutual Friends**: Use `Counter` + filtering to find meaningful connection suggestions.
- **Indexing**: `defaultdict` transforms O(n) searches into O(1) lookups.
- **Bucketing**: Raw averages hide patterns; grouping reveals actionable insights.
- **Rule-based Models**: Simple `if/elif` chains work for small data, but scale poorly.
- **Text Processing**: `str.lower()`, `split()`, and `Counter` are foundational NLP tools.

✅ **Next Step**: Run `exercises.ipynb` to solve Chapter 1 problems independently.